# Human Activity Recognition (HAR): Pipelines Comparison

This notebook reproduces and cleans the original HAR pipeline experiments:
- Baseline Gradient Boosting
- PCA + Gradient Boosting
- LDA + Gradient Boosting
- Hyperparameter tuning for LDA + Gradient Boosting
- LdaBoost model evaluation (from the local `LdaBoost` package)

All code, comments, and outputs are in English, with reproducible seeds and structured sections suitable for supplementary material.


In [22]:
# Reproducible setup and imports
import os
import random
import logging
from typing import Tuple, Dict

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, ShuffleSplit, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, make_scorer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier

import optuna
from optuna.samplers import TPESampler

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


# Global seed
RANDOM_SEED: int = 42
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Import LdaBoost from local package
import sys
REL_PROJECT_ROOT = "../../"
if REL_PROJECT_ROOT not in sys.path:
    sys.path.insert(0, REL_PROJECT_ROOT)
from LdaBoost import LdaBoost


## Dataset and preprocessing


In [23]:
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Sanitize column names to contain only letters, digits, and underscores.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame whose columns are to be sanitized.

    Returns
    -------
    pd.DataFrame
        DataFrame with sanitized column names.
    """
    df = df.copy()
    df.columns = [
        re.sub(r"[^A-Za-z0-9_]+", "_", col)  # type: ignore[name-defined]
        for col in df.columns
    ]
    return df

import re

# Load
train_df = pd.read_csv("Data/train.csv")
# Align test columns to train
test_df = pd.read_csv("Data/test.csv", names=list(train_df.columns), header=0)

# Clean names
train_df = clean_column_names(train_df)
test_df = clean_column_names(test_df)

# Split
X: pd.DataFrame = train_df.drop(columns="Activity")
y: pd.Series = train_df["Activity"]
X_test_raw: pd.DataFrame = test_df.drop(columns="Activity")
y_test_raw: pd.Series = test_df["Activity"]

# Encode labels (fit on train, transform test)
label_encoder = LabelEncoder()
y_encoded: np.ndarray = label_encoder.fit_transform(y)
y_test_encoded: np.ndarray = label_encoder.transform(y_test_raw)

# Feature list
features: list[str] = list(X.columns)

# CV splitter
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)



## Baseline: Gradient Boosting


In [24]:
# Preprocess: standardize all features
scaler_pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
])
base_preprocess = ColumnTransformer(
    remainder="passthrough",
    transformers=[("scale", scaler_pipe, features)],
)

base_boost = Pipeline(steps=[
    ("preprocess", base_preprocess),
    (
        "gb",
        GradientBoostingClassifier(
            n_estimators=1000,
            min_samples_leaf=4,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.6,
            max_features="sqrt",
            random_state=RANDOM_SEED,
        ),
    ),
])

acc_scores = cross_val_score(base_boost, X, y_encoded, cv=cv10, scoring="accuracy")
f1_scores = cross_val_score(base_boost, X, y_encoded, cv=cv10, scoring="f1_macro")

print(f"Accuracy (10-fold): mean={acc_scores.mean():.4f} std={acc_scores.std():.4f}")
print(f"F1 macro (10-fold): mean={f1_scores.mean():.4f} std={f1_scores.std():.4f}")

baseline_results: dict[str, np.ndarray] = {"accuracy": acc_scores, "f1_macro": f1_scores}



Accuracy (10-fold): mean=0.9946 std=0.0017
F1 macro (10-fold): mean=0.9949 std=0.0017


## PCA + Gradient Boosting


In [25]:
pca_pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=max(1, X.shape[1] // 16), random_state=RANDOM_SEED)),
])

pca_preprocess = ColumnTransformer(
    remainder="drop",
    transformers=[("pca", pca_pipe, features)],
)

pca_boost = Pipeline(steps=[
    ("preprocess", pca_preprocess),
    (
        "gb",
        GradientBoostingClassifier(
            n_estimators=1000,
            min_samples_leaf=4,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.6,
            max_features="sqrt",
            random_state=RANDOM_SEED,
        ),
    ),
])

acc_scores = cross_val_score(pca_boost, X, y_encoded, cv=cv10, scoring="accuracy")
f1_scores = cross_val_score(pca_boost, X, y_encoded, cv=cv10, scoring="f1_macro")

print(f"Accuracy (10-fold): mean={acc_scores.mean():.4f} std={acc_scores.std():.4f}")
print(f"F1 macro (10-fold): mean={f1_scores.mean():.4f} std={f1_scores.std():.4f}")

pca_results: dict[str, np.ndarray] = {"accuracy": acc_scores, "f1_macro": f1_scores}



Accuracy (10-fold): mean=0.9464 std=0.0094
F1 macro (10-fold): mean=0.9487 std=0.0087


## LDA + Gradient Boosting


In [26]:
class LDATransformer:
    """Wrapper around scikit-learn's LinearDiscriminantAnalysis to use in pipelines."""
    def __init__(self, n_components: int | None = None):
        self.n_components = n_components
        self.lda: LinearDiscriminantAnalysis | None = None

    def fit(self, X, y):
        self.lda = LinearDiscriminantAnalysis(n_components=self.n_components)
        self.lda.fit(X, y)
        return self

    def transform(self, X):
        assert self.lda is not None
        return self.lda.transform(X)

    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

lda_pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("lda", LDATransformer(n_components=5)),
])

lda_preprocess = ColumnTransformer(
    remainder="drop",
    transformers=[("lda", lda_pipe, list(range(X.shape[1])))],
)

lda_boost = Pipeline(steps=[
    ("preprocess", lda_preprocess),
    (
        "gb",
        GradientBoostingClassifier(
            n_estimators=1000,
            min_samples_leaf=4,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.6,
            max_features="sqrt",
            random_state=RANDOM_SEED,
        ),
    ),
])

acc_scores = cross_val_score(lda_boost, X, y_encoded, cv=cv10, scoring="accuracy")
f1_scores = cross_val_score(lda_boost, X, y_encoded, cv=cv10, scoring="f1_macro")

print(f"Accuracy (10-fold): mean={acc_scores.mean():.4f} std={acc_scores.std():.4f}")
print(f"F1 macro (10-fold): mean={f1_scores.mean():.4f} std={f1_scores.std():.4f}")

lda_results: dict[str, np.ndarray] = {"accuracy": acc_scores, "f1_macro": f1_scores}



Accuracy (10-fold): mean=0.9816 std=0.0040
F1 macro (10-fold): mean=0.9827 std=0.0038


## LdaBoost evaluation


In [21]:
# Convert features to numpy for the package model if needed
X_np: np.ndarray = X.to_numpy()

ldaboost_model = LdaBoost(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    random_state=RANDOM_SEED,
) 

acc_scores = []
f1_scores = []
for train_idx, test_idx in cv10.split(X_np, y_encoded):
    X_tr, X_te = X_np[train_idx], X_np[test_idx]
    y_tr, y_te = y_encoded[train_idx], y_encoded[test_idx]

    model = LdaBoost(
        n_estimators=ldaboost_model.n_estimators,
        learning_rate=ldaboost_model.learning_rate,
        max_depth=ldaboost_model.max_depth,
        random_state=RANDOM_SEED,
    )
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    acc_scores.append(accuracy_score(y_te, y_pred))
    f1_scores.append(f1_score(y_te, y_pred, average="macro"))

print(f"LdaBoost — Accuracy (10-fold): mean={np.mean(acc_scores):.4f} std={np.std(acc_scores):.4f}")
print(f"LdaBoost — F1 macro (10-fold): mean={np.mean(f1_scores):.4f} std={np.std(f1_scores):.4f}")



LdaBoost — Accuracy (10-fold): mean=0.9784 std=0.0060
LdaBoost — F1 macro (10-fold): mean=0.9795 std=0.0058
